# Rimworld-Statistics：回帰検証レポート

Colabブラウザ版で最新mainの **Core xUnitテスト、Pythonユニットテスト、合成セーブ取り込み**を実行して、匿名の検証レポートを生成します。CPU環境で動作し、RimWorld本体や実セーブは不要です。

**注意**：ここでの「PASS」は指定されたオフラインテストの結果です。実ゲーム・UI・TPSを保証しません。比較用CSVがあれば性能の変化を参考情報として表示します。Colab VMの性能は毎回変化するため、別セッションとの差を確定的な回帰と判断しません。

## 1. 設定

前回の `regression-benchmark.csv` があればColabにアップロードし、`REFERENCE_CSV`へフルパスを設定します。未入力でもテスト・グラフ・レポートは作成できます。

In [ ]:
REFERENCE_CSV = ""  # @param {type:"string"}
REGRESSION_SIGNAL_PERCENT = 20  # @param {type:"slider",min:5,max:100,step:5}
REPEATS = 3  # @param {type:"slider",min:1,max:10,step:1}
SAVE_COUNTS = (20, 55, 200)
PADDING_BYTES = 262144
assert 1 <= int(REPEATS) <= 10

## 2. ソースと.NET 10を準備

公開リポジトリのmainを読み取ります。Colabの一時領域にのみプログラムを作成し、GitHub側を変更しません。

In [ ]:
import io, json, os, subprocess, sys, time, urllib.request
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

REPO = Path("/content/rws-colab-regression-repository")
OUT = Path("/content/rws-colab-output")
OUT.mkdir(parents=True, exist_ok=True)

def execute(args, cwd=None, timeout=1800):
    started = time.monotonic()
    result = subprocess.run([str(a) for a in args], cwd=cwd, env=os.environ.copy(),
                            capture_output=True, text=True, errors="replace", timeout=timeout)
    return {"exit_code": result.returncode, "stdout": result.stdout,
            "stderr": result.stderr, "wall_seconds": round(time.monotonic() - started, 3)}

def checked(args, cwd=None, timeout=1800):
    result = execute(args, cwd=cwd, timeout=timeout)
    if result["exit_code"]:
        raise RuntimeError(f"コマンドが失敗しました (終了コード {result['exit_code']}): "
                           + result["stderr"][-1300:])
    return result

if not (REPO / ".git").is_dir():
    checked(["git", "clone", "--depth", "1",
             "https://github.com/pengin0503/Rimworld-Statistics.git", REPO], timeout=300)
else:
    checked(["git", "-C", REPO, "fetch", "--depth", "1", "origin", "main"], timeout=300)
    checked(["git", "-C", REPO, "checkout", "--detach", "FETCH_HEAD"])
dotnet_root = Path("/content/rws-dotnet")
if not (dotnet_root / "dotnet").is_file():
    installer = Path("/content/rws-dotnet-install.sh")
    urllib.request.urlretrieve("https://dot.net/v1/dotnet-install.sh", installer)
    checked(["bash", installer, "--channel", "10.0", "--install-dir", dotnet_root], timeout=900)
os.environ["DOTNET_ROOT"] = str(dotnet_root)
os.environ["PATH"] = str(dotnet_root) + os.pathsep + os.environ["PATH"]
os.environ["DOTNET_CLI_TELEMETRY_OPTOUT"] = "1"
os.environ["DOTNET_SKIP_FIRST_TIME_EXPERIENCE"] = "1"
sdk = checked(["dotnet", "--version"])["stdout"].strip()
assert sdk.startswith("10."), f".NET 10が必要です: {sdk}"
commit = checked(["git", "-C", REPO, "rev-parse", "HEAD"])["stdout"].strip()
print("対象コミット:", commit[:12], "/ .NET", sdk)

## 3. 既存の自動テストを実行

.NET CoreテストとPython検証の**終了コード**を記録します。失敗した場合は失敗扱いのまま結果を残します。コマンドのログをファイルへ残す場合は内容を確認してから公開してください。

In [ ]:
checks = []
build = execute(["dotnet", "build",
                 REPO / "Tests/RimworldStatistics.Core.Tests/RimworldStatistics.Core.Tests.csproj",
                 "-c", "Release", "-p:UseNuGetRefs=true"], cwd=REPO)
checks.append({"name": "Core tests build", "exit_code": build["exit_code"],
               "wall_seconds": build["wall_seconds"]})
if build["exit_code"] == 0:
    unit = execute(["dotnet", "test", "--project",
                    REPO / "Tests/RimworldStatistics.Core.Tests/RimworldStatistics.Core.Tests.csproj",
                    "-c", "Release", "--no-build"], cwd=REPO)
    checks.append({"name": "Core xUnit tests", "exit_code": unit["exit_code"],
                   "wall_seconds": unit["wall_seconds"]})
    print("Core xUnit (末尾のみ):")
    print("\n".join(unit["stdout"].splitlines()[-10:]))
else:
    checks.append({"name": "Core xUnit tests", "exit_code": None,
                   "wall_seconds": 0, "status": "not_run"})
python_tests = execute([sys.executable, "-m", "unittest", "discover",
                        "-s", "tools/tests", "-p", "test_*.py", "-v"], cwd=REPO)
checks.append({"name": "Python unit tests", "exit_code": python_tests["exit_code"],
               "wall_seconds": python_tests["wall_seconds"]})
print("Python tests:", "PASS" if python_tests["exit_code"] == 0 else "FAIL")
if any(check["exit_code"] not in (0,) for check in checks):
    print("失敗あり：後続のベンチマークは独立に実行できますが、全体判定はPASSになりません。")
display(pd.DataFrame(checks))

## 4. 同じVM内で合成セーブを測定

各件数で複数回取り込みを行い、ウォーム側がスキップされたか確認します。生成したCSVは次回のColabで比較参照に使えます。

In [ ]:
import csv

benchmark_project = REPO / "tools/SaveImportBenchmark/SaveImportBenchmark.csproj"
built = execute(["dotnet", "build", benchmark_project,
                 "-c", "Release", "-p:UseNuGetRefs=true"], cwd=REPO)
checks.append({"name": "SaveImportBenchmark build",
               "exit_code": built["exit_code"], "wall_seconds": built["wall_seconds"]})
benchmark_rows = []
if built["exit_code"] == 0:
    for count in SAVE_COUNTS:
        for repeat in range(1, REPEATS + 1):
            result = execute(
                ["dotnet", "run", "-c", "Release", "--project", benchmark_project,
                 "--no-build", "--no-launch-profile", "--", count, PADDING_BYTES], cwd=REPO)
            if result["exit_code"]:
                checks.append({"name": f"SaveImportBenchmark {count}/{repeat}",
                               "exit_code": result["exit_code"],
                               "wall_seconds": result["wall_seconds"]})
                continue
            parsed = list(csv.DictReader(io.StringIO(result["stdout"])))
            if len(parsed) != 1:
                checks.append({"name": f"SaveImportBenchmark {count}/{repeat}",
                               "exit_code": 1, "wall_seconds": result["wall_seconds"]})
                continue
            row = {k: float(v) if "." in v else int(v) for k, v in parsed[0].items()}
            if row["saves"] != count or row["home_partial"] != count:
                checks.append({"name": f"SaveImportBenchmark {count}/{repeat}",
                               "exit_code": 1, "wall_seconds": result["wall_seconds"]})
                continue
            row.update(repeat=repeat, commit=commit)
            benchmark_rows.append(row)
        print(f"{count}セーブ：{sum(x['saves']==count for x in benchmark_rows)}/{REPEATS}回成功")
if not benchmark_rows:
    raise RuntimeError("有効なベンチマーク測定がありません。先行するエラーを確認してください。")
benchmark = pd.DataFrame(benchmark_rows)
benchmark.to_csv(OUT / "regression-benchmark.csv", index=False)
display(benchmark[["saves", "repeat", "cold_ms", "warm_ms",
                   "cold_alloc_bytes", "warm_alloc_bytes"]].round(2))

## 5. 結果の可視化・前回との比較

前回と今回の比較は**異なるVM間の参考値**です。20%などの閾値超過を「注意信号」として示し、実測性能回帰と断定しません。

In [ ]:
current = (benchmark.groupby("saves", as_index=False)
           .agg(cold_median_ms=("cold_ms", "median"),
                warm_median_ms=("warm_ms", "median"),
                cold_alloc_median=("cold_alloc_bytes", "median")))
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(current["saves"], current["cold_median_ms"], marker="o", label="Cold")
ax.plot(current["saves"], current["warm_median_ms"], marker="s",
        linestyle="--", label="Warm")
ax.set(xlabel="Synthetic saves", ylabel="Median import time (ms)",
       title="Offline regression baseline (this Colab VM)")
ax.grid(alpha=.25)
ax.legend()
fig.tight_layout()
fig.savefig(OUT / "regression-latency.png", dpi=150)
plt.show()
plt.close(fig)

signals = []
if REFERENCE_CSV.strip():
    previous = pd.read_csv(Path(REFERENCE_CSV).expanduser())
    required = {"saves", "padding_bytes", "cold_ms"}
    if not required.issubset(previous.columns):
        raise ValueError("前回CSVに必要な列がありません。")
    previous = previous[previous["padding_bytes"] == PADDING_BYTES]
    baseline = previous.groupby("saves")["cold_ms"].median()
    for _, row in current.iterrows():
        count = int(row["saves"])
        if count not in baseline.index or float(baseline[count]) <= 0:
            continue
        change = 100 * (row["cold_median_ms"] / baseline[count] - 1)
        signals.append({"saves": count, "change_percent": round(change, 2),
                        "signal_only": change >= REGRESSION_SIGNAL_PERCENT})
    display(pd.DataFrame(signals))
    print("この比較はVM差を含むため、同一マシンで再実行するまで参考情報です。")
else:
    print("前回CSVなし：今回の結果を次回比較用に保存しました。")

## 6. JSONレポートと判定

全テストの終了コードと合成ベンチマークのみを出力します。テスト失敗を成功と表示しません。実ゲーム検証やMOD構成の検証は、別の検証ホストで実施してください。

In [ ]:
overall_pass = (all(item["exit_code"] == 0 for item in checks)
                and len(benchmark_rows) == len(SAVE_COUNTS) * REPEATS)
report = {
    "schema": 1, "commit": commit, "dotnet_sdk": sdk,
    "status": "PASS" if overall_pass else "FAIL",
    "checks": checks,
    "benchmark": {
        "counts": list(SAVE_COUNTS), "repeats": REPEATS,
        "padding_bytes": PADDING_BYTES,
        "medians": current.to_dict(orient="records"),
    },
    "cross_vm_signals_not_definitive_regressions": signals,
    "scope": "Offline Core xUnit, Python unittest and synthetic import; not in-game testing",
}
(OUT / "regression-report.json").write_text(
    json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
print("最終判定:", report["status"])
print("出力:", sorted(path.name for path in OUT.glob("regression-*")))
if not overall_pass:
    raise AssertionError("検証に失敗した項目があります。regression-report.jsonを確認してください。")